# Notebook 03: Prepare YOLO Data

## Purpose
Prepare all data required for YOLO training:
1. Convert COCO annotations to YOLO format
2. Create labels for synthetic images
3. Create training splits
4. Create data.yaml files
5. Validate everything

## Models
- **Model 1 (COCO Pretrained)**: Baseline - no additional training (yolov8s.pt)
- **Model 2 (Real-Only 8262)**: All 8,262 real training images
- **Model 3 (Real-Only 4131)**: 4,131 real images (fair comparison)
- **Model 4 (Mixed)**: 4,131 real + 4,131 synthetic images

## Test Sets
- Real Test: 1,547 images
- Synthetic Test: 1,547 images
- Mixed Test: 3,094 images

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import os
import shutil
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Imports complete")

## 2. Configuration

In [ ]:
# === CONFIGURATION ===
BASE_PATH = Path('/content/drive/MyDrive/StormVision')
DATA_PATH = BASE_PATH / 'data'

# Input paths
TRAIN_IMAGES = DATA_PATH / 'train' / 'images'
VAL_IMAGES = DATA_PATH / 'val' / 'images'
ANNOTATIONS = DATA_PATH / 'annotations'
SYNTHETIC_TRAIN_IMAGES = DATA_PATH / 'synthetic' / 'train' / 'images'
SYNTHETIC_TEST_IMAGES = DATA_PATH / 'synthetic' / 'test' / 'images'

# Output paths
PROCESSED = DATA_PATH / 'processed'
LABELS_DIR = PROCESSED / 'labels'
SPLITS_DIR = PROCESSED / 'splits'

# YOLO class mapping (skip 'ignored' class 0)
CATEGORY_MAPPING = {
    1: 0,  # swimmer
    2: 1,  # boat
    3: 2,  # jetski
    4: 3,  # life_saving_appliances
    5: 4,  # buoy
}

CLASS_NAMES = ['swimmer', 'boat', 'jetski', 'life_saving_appliances', 'buoy']

print(f"Base path: {BASE_PATH}")
print(f"Classes: {CLASS_NAMES}")

## 3. Verify Paths

In [ ]:
paths_to_check = [
    ('Base path', BASE_PATH),
    ('Train images', TRAIN_IMAGES),
    ('Val images', VAL_IMAGES),
    ('Annotations', ANNOTATIONS),
    ('Synthetic train images', SYNTHETIC_TRAIN_IMAGES),
    ('Synthetic test images', SYNTHETIC_TEST_IMAGES),
]

print("Checking paths:")
all_ok = True
for name, path in paths_to_check:
    exists = path.exists()
    if exists and path.is_dir():
        count = len(list(path.glob('*')))
    else:
        count = 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {name}: {path} ({count} files)")
    if not exists:
        all_ok = False

if all_ok:
    print("\nAll paths verified.")
else:
    print("\nSome paths missing! Check configuration.")

## 4. Load COCO Annotations

In [ ]:
def load_coco_annotations(json_path):
    """Load COCO format annotations and create lookup dictionaries."""
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    images = {img['id']: img for img in data['images']}
    
    annotations_by_image = {}
    for ann in data['annotations']:
        img_id = ann['image_id']
        if img_id not in annotations_by_image:
            annotations_by_image[img_id] = []
        annotations_by_image[img_id].append(ann)
    
    categories = {cat['id']: cat['name'] for cat in data['categories']}
    
    return images, annotations_by_image, categories

print("Loading train annotations...")
train_images, train_annotations, categories = load_coco_annotations(
    ANNOTATIONS / 'instances_train.json'
)
print(f"  Loaded {len(train_images)} images, {sum(len(v) for v in train_annotations.values())} annotations")

print("Loading val annotations...")
val_images, val_annotations, _ = load_coco_annotations(
    ANNOTATIONS / 'instances_val.json'
)
print(f"  Loaded {len(val_images)} images, {sum(len(v) for v in val_annotations.values())} annotations")

print(f"\nCategories: {categories}")

## 5. COCO to YOLO Conversion Functions

In [ ]:
def coco_to_yolo(bbox, img_width, img_height):
    """
    Convert COCO bbox [x, y, width, height] to YOLO format [x_center, y_center, width, height].
    All values normalized to 0-1.
    """
    x, y, w, h = bbox
    
    x_center = (x + w / 2) / img_width
    y_center = (y + h / 2) / img_height
    w_norm = w / img_width
    h_norm = h / img_height
    
    # Clamp values to [0, 1]
    x_center = max(0, min(1, x_center))
    y_center = max(0, min(1, y_center))
    w_norm = max(0, min(1, w_norm))
    h_norm = max(0, min(1, h_norm))
    
    return x_center, y_center, w_norm, h_norm


def create_yolo_label(annotations, img_info, category_mapping):
    """
    Create YOLO format label string from COCO annotations.
    Returns list of lines: 'class_id x_center y_center width height'
    """
    lines = []
    img_w = img_info['width']
    img_h = img_info['height']
    
    for ann in annotations:
        cat_id = ann['category_id']
        
        if cat_id not in category_mapping:
            continue
        
        yolo_class = category_mapping[cat_id]
        x_c, y_c, w, h = coco_to_yolo(ann['bbox'], img_w, img_h)
        
        line = f"{yolo_class} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}"
        lines.append(line)
    
    return lines


# Test conversion
test_img_id = list(train_images.keys())[0]
test_img_info = train_images[test_img_id]
test_anns = train_annotations.get(test_img_id, [])
test_labels = create_yolo_label(test_anns, test_img_info, CATEGORY_MAPPING)

print(f"Test conversion for image {test_img_info['file_name']}:")
print(f"  Image size: {test_img_info['width']}x{test_img_info['height']}")
print(f"  Annotations: {len(test_anns)}")
print(f"  YOLO labels:")
for label in test_labels[:3]:
    print(f"    {label}")

## 6. Create Label Directories

In [ ]:
label_dirs = {
    'train': LABELS_DIR / 'train',
    'val': LABELS_DIR / 'val',
    'synthetic_train': LABELS_DIR / 'synthetic_train',
    'synthetic_test': LABELS_DIR / 'synthetic_test',
}

for name, path in label_dirs.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"Created: {path}")

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Created: {SPLITS_DIR}")

## 7. Generate Labels for Real Images

In [ ]:
def save_labels(images_dict, annotations_dict, output_dir, category_mapping):
    """
    Save YOLO format labels for all images.
    Returns dict mapping filename to label path.
    """
    saved = {}
    empty_count = 0
    
    for img_id, img_info in tqdm(images_dict.items(), desc="Creating labels"):
        filename = img_info['file_name']
        basename = Path(filename).stem
        
        anns = annotations_dict.get(img_id, [])
        labels = create_yolo_label(anns, img_info, category_mapping)
        
        label_path = output_dir / f"{basename}.txt"
        
        with open(label_path, 'w') as f:
            f.write('\n'.join(labels))
        
        saved[basename] = label_path
        
        if len(labels) == 0:
            empty_count += 1
    
    return saved, empty_count

print("Creating labels for TRAIN images...")
train_labels_map, train_empty = save_labels(
    train_images, train_annotations, label_dirs['train'], CATEGORY_MAPPING
)
print(f"  Created {len(train_labels_map)} labels ({train_empty} empty)")

print("\nCreating labels for VAL images...")
val_labels_map, val_empty = save_labels(
    val_images, val_annotations, label_dirs['val'], CATEGORY_MAPPING
)
print(f"  Created {len(val_labels_map)} labels ({val_empty} empty)")

## 8. Generate Labels for Synthetic Images

In [ ]:
def create_synthetic_labels(synth_images_dir, source_labels_dir, output_dir):
    """
    Create labels for synthetic images by copying from original image labels.
    Synthetic images are named like: 1234_synth.jpg
    Original labels are named like: 1234.txt
    Supports both .jpg and .png formats.
    """
    # Support both jpg and png
    synth_images = list(synth_images_dir.glob('*.jpg')) + list(synth_images_dir.glob('*.png'))
    created = 0
    errors = []
    
    for synth_img in tqdm(synth_images, desc="Creating synthetic labels"):
        synth_basename = synth_img.stem  # e.g., '1234_synth'
        original_id = synth_basename.replace('_synth', '')  # e.g., '1234'
        
        source_label = source_labels_dir / f"{original_id}.txt"
        
        if not source_label.exists():
            errors.append(original_id)
            continue
        
        dest_label = output_dir / f"{synth_basename}.txt"
        shutil.copy(source_label, dest_label)
        created += 1
    
    return created, errors

print("Creating labels for SYNTHETIC TRAIN images...")
synth_train_created, synth_train_errors = create_synthetic_labels(
    SYNTHETIC_TRAIN_IMAGES,
    label_dirs['train'],
    label_dirs['synthetic_train']
)
print(f"  Created {synth_train_created} labels")
if synth_train_errors:
    print(f"  Errors (missing original): {len(synth_train_errors)}")

print("\nCreating labels for SYNTHETIC TEST images...")
synth_test_created, synth_test_errors = create_synthetic_labels(
    SYNTHETIC_TEST_IMAGES,
    label_dirs['val'],
    label_dirs['synthetic_test']
)
print(f"  Created {synth_test_created} labels")
if synth_test_errors:
    print(f"  Errors (missing original): {len(synth_test_errors)}")

## 9. Copy Labels to YOLO Expected Locations

YOLO expects labels in a 'labels' folder parallel to 'images' folder.

In [ ]:
copy_targets = [
    (label_dirs['train'], DATA_PATH / 'train' / 'labels'),
    (label_dirs['val'], DATA_PATH / 'val' / 'labels'),
    (label_dirs['synthetic_train'], DATA_PATH / 'synthetic' / 'train' / 'labels'),
    (label_dirs['synthetic_test'], DATA_PATH / 'synthetic' / 'test' / 'labels'),
]

print("Copying labels to YOLO expected locations:")
for source, target in copy_targets:
    target.mkdir(parents=True, exist_ok=True)
    
    # Copy all label files
    count = 0
    for label_file in source.glob('*.txt'):
        shutil.copy(label_file, target / label_file.name)
        count += 1
    
    print(f"  {target}: {count} labels copied")

print("\n[OK] Labels copied to YOLO locations.")

## 10. Count Available Images

In [ ]:
# Get all available images (support both jpg and png)
real_train_images = sorted([str(p) for p in TRAIN_IMAGES.glob('*.jpg')])
real_test_images = sorted([str(p) for p in VAL_IMAGES.glob('*.jpg')])

# Synthetic - support both formats
synth_train_jpg = [str(p) for p in SYNTHETIC_TRAIN_IMAGES.glob('*.jpg')]
synth_train_png = [str(p) for p in SYNTHETIC_TRAIN_IMAGES.glob('*.png')]
synthetic_train_images = sorted(synth_train_jpg + synth_train_png)

synth_test_jpg = [str(p) for p in SYNTHETIC_TEST_IMAGES.glob('*.jpg')]
synth_test_png = [str(p) for p in SYNTHETIC_TEST_IMAGES.glob('*.png')]
synthetic_test_images = sorted(synth_test_jpg + synth_test_png)

print("Available images:")
print(f"  Real train: {len(real_train_images)}")
print(f"  Real test (val): {len(real_test_images)}")
print(f"  Synthetic train: {len(synthetic_train_images)} ({len(synth_train_jpg)} jpg, {len(synth_train_png)} png)")
print(f"  Synthetic test: {len(synthetic_test_images)} ({len(synth_test_jpg)} jpg, {len(synth_test_png)} png)")

## 11. Create Training Splits

In [ ]:
random.seed(SEED)

num_synthetic = len(synthetic_train_images)

# Split 1: Real Only (all real images)
real_only_sample = real_train_images.copy()

# Split 2: Real 4131 (same count as synthetic for fair comparison)
real_4131_sample = random.sample(real_train_images, min(num_synthetic, len(real_train_images)))

# Split 3: Mixed (4131 real + 4131 synthetic)
mixed_real_sample = real_4131_sample.copy()
mixed_synthetic_sample = synthetic_train_images.copy()
mixed_all = mixed_real_sample + mixed_synthetic_sample
random.shuffle(mixed_all)

print("Split configuration:")
print(f"  Real-Only (all): {len(real_only_sample)} images")
print(f"  Real-4131 (fair): {len(real_4131_sample)} images")
print(f"  Mixed: {len(mixed_real_sample)} real + {len(mixed_synthetic_sample)} synthetic = {len(mixed_all)} images")

## 12. Save Split Files

In [ ]:
def save_split(filename, data):
    path = SPLITS_DIR / filename
    with open(path, 'w') as f:
        f.write('\n'.join(data))
    print(f"  Saved {filename}: {len(data)} entries")

print("Saving split files:")
save_split('train_real_only.txt', real_only_sample)
save_split('train_real_4131.txt', real_4131_sample)
save_split('train_mixed.txt', mixed_all)
save_split('test_real.txt', real_test_images)
save_split('test_synthetic.txt', synthetic_test_images)
save_split('test_mixed.txt', real_test_images + synthetic_test_images)

## 13. Create data.yaml Files

In [ ]:
def create_yaml(name, train_file, val_file, output_path):
    """Create YOLO data.yaml file."""
    yaml_content = f"""# {name}
path: {DATA_PATH}
train: {train_file}
val: {val_file}

nc: {len(CLASS_NAMES)}

names:
"""
    for i, cls_name in enumerate(CLASS_NAMES):
        yaml_content += f"  {i}: {cls_name}\n"
    
    with open(output_path, 'w') as f:
        f.write(yaml_content)
    
    print(f"  Created {output_path.name}")

print("Creating data.yaml files:")

# Real-Only (all images)
create_yaml(
    "StormVision Real-Only (All)",
    SPLITS_DIR / 'train_real_only.txt',
    SPLITS_DIR / 'test_real.txt',
    PROCESSED / 'yolo_real.yaml'
)

# Real-4131 (for fair comparison)
create_yaml(
    "StormVision Real-Only (4131)",
    SPLITS_DIR / 'train_real_4131.txt',
    SPLITS_DIR / 'test_mixed.txt',
    PROCESSED / 'yolo_real_4131.yaml'
)

# Mixed
create_yaml(
    "StormVision Mixed (4131 Real + 4131 Synthetic)",
    SPLITS_DIR / 'train_mixed.txt',
    SPLITS_DIR / 'test_mixed.txt',
    PROCESSED / 'yolo_mixed.yaml'
)

In [ ]:
# Display yaml contents
print("=" * 50)
print("yolo_real.yaml")
print("=" * 50)
print(open(PROCESSED / 'yolo_real.yaml').read())

print("\n" + "=" * 50)
print("yolo_real_4131.yaml")
print("=" * 50)
print(open(PROCESSED / 'yolo_real_4131.yaml').read())

print("\n" + "=" * 50)
print("yolo_mixed.yaml")
print("=" * 50)
print(open(PROCESSED / 'yolo_mixed.yaml').read())

## 14. Validation

In [ ]:
def validate_split(split_file, label_dirs_list):
    """Validate that every image in split has a corresponding label."""
    with open(split_file, 'r') as f:
        images = [line.strip() for line in f if line.strip()]
    
    matched = 0
    missing = []
    
    for img_path in images:
        basename = Path(img_path).stem
        
        found = False
        for label_dir in label_dirs_list:
            if (label_dir / f"{basename}.txt").exists():
                found = True
                break
        
        if found:
            matched += 1
        else:
            missing.append(basename)
    
    return len(images), matched, missing

print("Validating image-label matching:")

splits_to_validate = [
    ('train_real_only.txt', [DATA_PATH / 'train' / 'labels']),
    ('train_real_4131.txt', [DATA_PATH / 'train' / 'labels']),
    ('train_mixed.txt', [DATA_PATH / 'train' / 'labels', DATA_PATH / 'synthetic' / 'train' / 'labels']),
    ('test_real.txt', [DATA_PATH / 'val' / 'labels']),
    ('test_synthetic.txt', [DATA_PATH / 'synthetic' / 'test' / 'labels']),
    ('test_mixed.txt', [DATA_PATH / 'val' / 'labels', DATA_PATH / 'synthetic' / 'test' / 'labels']),
]

all_valid = True
for split_name, label_dirs_list in splits_to_validate:
    split_path = SPLITS_DIR / split_name
    if not split_path.exists():
        print(f"  [MISSING] {split_name}")
        all_valid = False
        continue
    
    total, matched, missing = validate_split(split_path, label_dirs_list)
    status = "OK" if matched == total else "WARN"
    print(f"  [{status}] {split_name}: {matched}/{total} matched")
    if missing:
        all_valid = False
        print(f"         Missing: {len(missing)} (first 3: {missing[:3]})")

if all_valid:
    print("\n[OK] All validations passed!")
else:
    print("\n[WARN] Some labels missing - check errors above.")

In [ ]:
def validate_label_format(label_dir, sample_size=100):
    """Validate YOLO label format."""
    label_files = list(Path(label_dir).glob('*.txt'))
    
    if not label_files:
        return 0, ["No label files found"]
    
    errors = []
    valid = 0
    sample = random.sample(label_files, min(sample_size, len(label_files)))
    
    for label_file in sample:
        try:
            with open(label_file, 'r') as f:
                lines = f.readlines()
            
            for i, line in enumerate(lines):
                line = line.strip()
                if not line:
                    continue
                
                parts = line.split()
                if len(parts) != 5:
                    errors.append(f"{label_file.name}: Line {i+1} has {len(parts)} parts")
                    continue
                
                class_id = int(parts[0])
                coords = [float(x) for x in parts[1:]]
                
                if class_id < 0 or class_id >= len(CLASS_NAMES):
                    errors.append(f"{label_file.name}: Invalid class {class_id}")
                
                for c in coords:
                    if c < 0 or c > 1:
                        errors.append(f"{label_file.name}: Coordinate out of range")
            
            valid += 1
        except Exception as e:
            errors.append(f"{label_file.name}: {str(e)}")
    
    return valid, errors

print("\nValidating label format:")
label_locations = [
    ('train', DATA_PATH / 'train' / 'labels'),
    ('val', DATA_PATH / 'val' / 'labels'),
    ('synthetic_train', DATA_PATH / 'synthetic' / 'train' / 'labels'),
    ('synthetic_test', DATA_PATH / 'synthetic' / 'test' / 'labels'),
]

for name, path in label_locations:
    if not path.exists():
        print(f"  [MISSING] {name}: {path}")
        continue
    valid, errors = validate_label_format(path)
    total = len(list(path.glob('*.txt')))
    status = "OK" if not errors else "WARN"
    print(f"  [{status}] {name}: {total} labels, {valid} sampled OK, {len(errors)} errors")
    if errors:
        for err in errors[:2]:
            print(f"       {err}")

## 15. Summary

In [ ]:
print("=" * 60)
print("                    PREPARATION COMPLETE")
print("=" * 60)

print("\nLabel Directories:")
label_locations = [
    ('train', DATA_PATH / 'train' / 'labels'),
    ('val', DATA_PATH / 'val' / 'labels'),
    ('synthetic/train', DATA_PATH / 'synthetic' / 'train' / 'labels'),
    ('synthetic/test', DATA_PATH / 'synthetic' / 'test' / 'labels'),
]
for name, path in label_locations:
    count = len(list(path.glob('*.txt'))) if path.exists() else 0
    print(f"  {name}: {count} labels")

print("\nSplit Files:")
for f in sorted(SPLITS_DIR.glob('*.txt')):
    with open(f, 'r') as file:
        count = len(file.readlines())
    print(f"  {f.name}: {count} images")

print("\nYAML Files:")
for f in sorted(PROCESSED.glob('*.yaml')):
    print(f"  {f.name}")

print("\nModels:")
print(f"  Model 0 (COCO):       Pretrained baseline (no training)")
print(f"  Model 1 (Real_8262):  {len(real_only_sample)} images")
print(f"  Model 2 (Real_4131):  {len(real_4131_sample)} images")
print(f"  Model 3 (Mixed):      {len(mixed_all)} images ({len(mixed_real_sample)} real + {len(mixed_synthetic_sample)} synth)")

print("\nTest Sets:")
print(f"  Real:      {len(real_test_images)} images")
print(f"  Synthetic: {len(synthetic_test_images)} images")
print(f"  Mixed:     {len(real_test_images) + len(synthetic_test_images)} images")

print("\n" + "=" * 60)
print("Next: Run 04_train_and_evaluate.ipynb")
print("=" * 60)